# Create EDCTP Awards

Creates OpenAlex award rows from EDCTP's own official public grants system (the EDCTP2 grants portal).

**Data source:** `https://www.edctpgrants.org/publicportal` — an AngularJS SmartSimple grants system backed by the JSON endpoint `GET /publicportal/search?pageSize=N&pageIndex=P` (response guarded by the AngularJS `)]}',` prefix, stripped by the scraper). EDCTP's edctp.org project portal is browsable HTML only (no export), and **CORDIS holds ONLY the umbrella EDCTP2 programme grant, not the individual sub-grants** — so the per-grant data only exists on this portal.
**S3 location:** `s3a://openalex-ingest/awards/edctp/edctp_projects.parquet`
**OpenAlex funder:** `F4320338462` — EDCTP (European & Developing Countries Clinical Trials Partnership), ROR `https://ror.org/031jv9v19`, DOI `10.13039/501100001713`, country `NL`. F4320* → Path A (the `openalex.common.funder` dim should carry it; a hardcoded COALESCE fallback guards the Abel/MinCiencias canonical-funder gap anyway).
**Provenance:** `edctp_grants_portal`.
**Priority:** 205.

**Tier 3** — EDCTP publishes EUR grant amounts (~86% coverage in the public feed), so the Step 6.7 amount check is **NOT waived**.

**Schema choices:**
- One row per EDCTP grant (deduped on the portal GUID by the scraper). Stable `funder_award_id = reference` (e.g. `RIA2018D-2496`).
- `amount` parsed from the `totalAward` string by the scraper (the feed's `totalAwardDecimal` is always 0.0 and is unused). `currency = EUR`.
- `funding_type`: `fellowship` for the TMA (Training & Mobility Actions) scheme family, else `research` (computed by the scraper from the reference prefix). `funder_scheme` = the reference's alpha prefix (RIA / TMA / CSA / SRIA / …).
- `lead_investigator` from `leadApplicantName` (coordinator/PI; leading titles + trailing post-nominals stripped and split into given/family **in the scraper**, per runbook §2.4.1 — the notebook uses the pre-split columns and does no SQL name-splitting). Affiliation = the host/lead institution with its country (parsed out of `hostOrganisationName`, including formal names like `Tanzania, United Republic of`).
- `description` from the project `publicSummary` (HTML stripped to text by the scraper).
- `start_date`/`end_date` from the portal ISO dates; `start_year`/`end_year` derived.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.edctp_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/edctp/edctp_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS total_edctp_projects FROM openalex.awards.edctp_raw;

## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `grant_guid`, `acronym`, `display_name`, `description`, `host_organisation_raw`, `host_institution`, `host_country`, `lead_name_raw`, `lead_given_name`, `lead_family_name`, `total_award_raw`, `amount`, `currency`, `funding_type`, `funder_scheme`, `start_date`, `end_date`, `start_year`, `end_year`, `duration_months`, `project_website`, `landing_page_url`, `downloaded_at`.

`amount` is already a clean numeric string parsed from `totalAward` (with `total_award_raw` kept for audit). `currency` is `EUR`.

In [ ]:
%sql
DESCRIBE openalex.awards.edctp_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, lead_given_name, lead_family_name,
       host_institution, host_country, total_award_raw, amount, currency,
       funding_type, funder_scheme, start_date, end_date, landing_page_url
FROM openalex.awards.edctp_raw
LIMIT 10;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(lead_family_name) AS has_lead,
    COUNT(host_institution) AS has_institution,
    COUNT(host_country) AS has_country,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.edctp_raw;

## Step 1.6: Fail-Fast — Verify EDCTP Funder Exists

`F4320338462` is an F4320* funder (Path A), so `openalex.common.funder` should return exactly one row. The transform still CROSS JOINs a `funder_resolved` CTE with a hardcoded COALESCE fallback so the awards cannot silently drop to zero if the dim is missing the row (the Abel/MinCiencias canonical-funder gap pattern).

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320338462;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.edctp_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320338462 AS funder_id,
        COALESCE(MAX(display_name), 'European & Developing Countries Clinical Trials Partnership') AS display_name,
        COALESCE(MAX(ror_id), 'https://ror.org/031jv9v19') AS ror_id,
        COALESCE(MAX(doi), '10.13039/501100001713') AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320338462
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        CASE WHEN TRY_CAST(end_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(end_year AS INT) END AS end_year_int,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_date_d,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_date_d,
        NULLIF(TRIM(host_institution), '') AS leader_affiliation_name,
        NULLIF(TRIM(host_country), '') AS leader_affiliation_country,
        NULLIF(TRIM(lead_given_name), '') AS leader_given_name,
        NULLIF(TRIM(lead_family_name), '') AS leader_family_name
    FROM openalex.awards.edctp_raw
)
SELECT
    abs(xxhash64(CONCAT('4320338462:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name,
    NULLIF(TRIM(rp.description), '') AS description,
    4320338462 AS funder_id,
    rp.funder_award_id,
    rp.amount_double AS amount,
    CASE WHEN rp.amount_double IS NOT NULL THEN 'EUR' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    COALESCE(NULLIF(TRIM(rp.funding_type), ''), 'research') AS funding_type,
    NULLIF(TRIM(rp.funder_scheme), '') AS funder_scheme,
    'edctp_grants_portal' AS provenance,
    rp.start_date_d AS start_date,
    rp.end_date_d AS end_date,
    rp.start_year_int AS start_year,
    rp.end_year_int AS end_year,
    CASE
        WHEN rp.leader_family_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                rp.leader_given_name AS given_name,
                rp.leader_family_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    rp.leader_affiliation_country AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320338462:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name IS NOT NULL;

## Step 3: Insert into `openalex_awards_raw` at Priority 205

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'edctp_grants_portal' AND priority = 205;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    205 AS priority
FROM openalex.awards.edctp_awards;

## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_edctp_awards FROM openalex.awards.edctp_awards;

In [ ]:
%sql
-- Duplicate funder_award_id / id must be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.edctp_awards;

In [ ]:
%sql
-- Completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    SUM(CASE WHEN lead_investigator.affiliation.country IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_country,
    COUNT(start_year) AS has_start_year,
    COUNT(start_date) AS has_start_date,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_name
FROM openalex.awards.edctp_awards;

### Step 6.7: Amount coverage — NOT waived

EDCTP publishes EUR grant amounts, so amount coverage must be substantial (expected ~85%). This is a Tier-3 ingest; the amount check is **not** waived. `pct_amount` well above 50% is the pass condition.

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount, percentile_approx(amount, 0.5) AS median_amount,
    ROUND(SUM(amount), 0) AS total_eur
FROM openalex.awards.edctp_awards;

In [ ]:
%sql
-- Hard gate: fail the run if amount coverage falls below 50% (Step 6.7, not waived).
SELECT assert_true(
    try_divide(COUNT(amount), COUNT(*)) >= 0.5,
    'EDCTP amount coverage below 50% -- investigate before integrating (Step 6.7 not waived).'
) AS amount_check_passed
FROM openalex.awards.edctp_awards;

In [ ]:
%sql
-- PI frequency (Step 6.4a): no name should dominate; no institution-as-name.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.edctp_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Funding-type + scheme distribution sanity.
SELECT funding_type, funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS eur_total
FROM openalex.awards.edctp_awards
GROUP BY funding_type, funder_scheme ORDER BY awards DESC;

In [ ]:
%sql
SELECT id, funder_award_id, SUBSTRING(display_name, 1, 70) AS title, amount, currency,
       start_year, end_year,
       lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS affiliation,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.edctp_awards
ORDER BY start_year DESC, id LIMIT 20;

In [ ]:
%sql
-- Confirm the priority insert landed.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'edctp_grants_portal'
GROUP BY priority, provenance;